# SOXX Series ID 检索页零网络解析

只读上一阶段已由 NAS 恢复核验的两个 SEC 搜索页，抽取系列专属 N-Q/NPORT-P accession 与申报日；与信托目录已核验 accession 对照，不打开正文。


In [1]:
import json,hashlib,re
from pathlib import Path
from bs4 import BeautifulSoup
root=Path.cwd();card=root/'research/stage1_soxx_series_filing_search_20261023';plan=json.loads((card/'plan.json').read_text());receipt=json.loads((card/'capture_receipts.json').read_text())
raw=root/'data/snapshots/stage1_soxx_series_filing_search_20261023'
source={x['form']:x for x in receipt};rows=[]
for form,name in [('N-Q','N_Q.html'),('NPORT-P','NPORT_P.html')]:
 path=raw/name;assert hashlib.sha256(path.read_bytes()).hexdigest()==source[form]['raw_sha256']
 soup=BeautifulSoup(path.read_bytes(),'html.parser');identity=soup.select_one('p.identInfo')
 assert identity is not None and 'S000004354' in identity.get_text(' ',strip=True) and 'Semiconductor ETF' in identity.get_text(' ',strip=True)
 table=soup.select_one('#seriesDiv table.tableFile2');assert table is not None
 selected=[]
 for tr in table.select('tr'):
  cells=tr.find_all('td',recursive=False)
  if not cells:continue
  assert len(cells)==5
  item_form=cells[0].get_text(' ',strip=True)
  assert item_form in json.loads((card/'parse_repair_plan.json').read_text())['allowed_forms_by_page'][form],(form,item_form)
  link=cells[1].find('a',id='documentsbutton');assert link and link.get('href')
  match=re.search(r'([0-9]{10}-[0-9]{2}-[0-9]{6})-index\.htm',link['href'])
  assert match
  accession=match.group(1);filed=cells[3].get_text(' ',strip=True)
  assert re.fullmatch(r'20[0-9]{2}-[0-9]{2}-[0-9]{2}',filed)
  selected.append({'form':item_form,'accession':accession,'filed':filed,'index_href':link['href']})
 assert len(selected)==source[form]['accession_numbers_unique']
 rows.extend(selected)
primary=json.loads((root/'research/stage1_soxx_legacy_basket_history_20261023/candidate_trust_filings.json').read_text())['rows']
primary_accessions={x['accessionNumber'] for x in primary}
nq_2011_2018=[x for x in rows if x['form']=='N-Q' and '2011'<=x['filed'][:4]<='2018']
assert all(x['accession'] in primary_accessions for x in nq_2011_2018)
assert any(x['accession']=='0001193125-11-176858' for x in rows)
assert not any(x['accession'] in {'0001193125-11-086451','0001193125-11-086453'} for x in rows)
result={'status':'SEC_series_specific_filing_index_identity_and_accession_readback_passed','source_NAS_archive_sha256':json.loads((card/'result.json').read_text())['nas_archive_sha256'],'series_id':'S000004354','fund_heading':'iShares Semiconductor ETF','N_Q_rows':sum(x['form']=='N-Q' for x in rows),'NPORT_P_rows':sum(x['form']=='NPORT-P' for x in rows),'NPORT_P_amendment_rows':sum(x['form']=='NPORT-P/A' for x in rows),'N_Q_2011_2018_rows':len(nq_2011_2018),'N_Q_2011_2018_all_in_primary_trust_inventory':True,'known_2011_Apr_positive_present':True,'two_known_2011_Jan_negative_absent':True,'rows':rows,'SOXX_holdings_values_qualified':False,'acceptance_timing_qualified_from_this_page':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'series_index_parse.json').write_text(json.dumps(result,indent=2)+'\n')
print({k:result[k] for k in ('status','N_Q_rows','NPORT_P_rows','N_Q_2011_2018_rows','known_2011_Apr_positive_present','two_known_2011_Jan_negative_absent')})


{'status': 'SEC_series_specific_filing_index_identity_and_accession_readback_passed', 'N_Q_rows': 27, 'NPORT_P_rows': 28, 'N_Q_2011_2018_rows': 15, 'known_2011_Apr_positive_present': True, 'two_known_2011_Jan_negative_absent': True}


## 中文结论

SEC SOXX Series `S000004354` 搜索页核对通过：N-Q 共 27 条，其中 2011–2018 为 15 条；NPORT-P 28 条、修订 NPORT-P/A 1 条。15 条旧 N-Q 全与信托目录 accession 对上；2011 年 4 月已核实的 SOXX 正文在列，两条 1 月非 SOXX 正文不在。原页面经 NAS 归档恢复，SHA-256 `14789c4727e1a518511e56dbc6112a975798a2dc0efc2def0e234de54f8bdc90`。一次离线解析补充允许 NPORT-P/A，保留失败回执，没有增加网络请求。仅为系列来源索引，0 正文、0 收益、0 因子、0 拟合、0 账户；完整持仓与首次可用时钟待核对。
